# Exploratory Data Analysis of House Prices

## Introduction

Exploratory Data Analysis (EDA) is the process of analysing and understanding data using statistics and visualisations. In this project we explore the **Ames Housing dataset** to understand which factors drive house sale prices. This is an EDA-only project: **no machine learning model is built**.

## Objective

To analyse house prices using Python and understand which factors affect house sale prices, while practising the full EDA workflow.

## Guiding Questions

These questions were written down *before* touching the data. Every section below helps answer one or more of them.

1. How is `SalePrice` distributed? Is it skewed?
2. Which numerical features correlate most strongly with `SalePrice`?
3. Does overall quality (`OverallQual`) change price in a clear, ordered way?
4. Do some neighbourhoods consistently cost more than others?
5. Are there outliers, and are they errors or real but unusual sales?
6. Which columns have missing values, and which of those are not truly missing?

## Dataset

*House Prices: Advanced Regression Techniques* (Kaggle), file `train.csv`: 1460 rows and 81 columns.

## Tools Used

Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy, Jupyter Notebook.

## Notebook Structure

| Section | Content |
|---|---|
| 1 | Data loading and first look |
| 2 | Classification of variables |
| 3 | Descriptive statistics |
| 4 | Distribution analysis (univariate) |
| 5-7 | Correlation, scatter/pair plots, heatmap (bivariate and multivariate) |
| 8 | Outlier analysis |
| 9 | Data quality checks and cleaning |
| 10 | Final dashboard |
| 11 | Verifying the expected insights |
| 12 | Conclusions and limitations |

**Why cleaning comes at Step 9:** Steps 1-8 explore the *raw* data (`df`) so we can see the problems first (missing values, outliers). Step 9 builds a separate `df_clean`, and the dashboard in Step 10 uses `df_clean`. The raw `df` is never overwritten.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.ticker import FuncFormatter
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# One colour palette used across the whole notebook
BASE   = "#2a6f97"   # main colour
ACCENT = "#e07a5f"   # highlight colour (outliers, "No", etc.)
GREEN  = "#3a7d44"   # secondary highlight (median lines)
thousands = FuncFormatter(lambda x, _: f"{x/1000:,.0f}k")

print("Libraries imported successfully!")

## 1. Dataset Loading and First Look

**What we do:** load the dataset and check its shape, data types and missing values.

In [ ]:
df = pd.read_csv("train.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

In [ ]:
df.info()

#### Checking the missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_table = pd.DataFrame({
    "missing": missing,
    "percent": (missing / len(df) * 100).round(2)
})
display(missing_table)

print(f"\nRows: {df.shape[0]}  |  Columns: {df.shape[1]}")
print("Total Columns with Missing Values:", len(missing_table))
print("Columns missing in more than 50% of rows:",
      missing_table[missing_table["percent"] > 50].index.tolist())

### What we learned

- The dataset has **1460 rows and 81 columns**, a mix of numerical and categorical variables.
- **19 columns contain missing values.** A few of them (`PoolQC`, `MiscFeature`, `Alley`, `Fence`) are missing for most rows. A house that simply has no pool or no alley is very unlikely to be "unknown", so we suspect that in this dataset a missing value often means **"feature absent"**. We will confirm this in Step 9 instead of deleting these columns.
- `MasVnrType` shows a large number of missing values too. We will look at that column closely in Step 9, because pandas reads the text `"None"` as a missing value.

## 2. Classification of Variables

Variables are classified into three main types:

1. **Numerical** (continuous and discrete): numbers that can be measured or counted.
2. **Categorical** (nominal): groups with no natural order.
3. **Ordinal**: groups with a meaningful order (e.g. Poor < Fair < Typical < Good < Excellent).

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

print("Total Numerical:", len(numeric_cols))
print("Total Categorical:", len(categorical_cols))
print("\nNumerical Columns:\n", list(numeric_cols))
print("\nCategorical Columns:\n", list(categorical_cols))

#### Ordinal variables

In [ ]:
quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]      # Poor < Fair < Typical < Good < Excellent

ordinal_map = {
    "ExterQual":   quality_order,
    "KitchenQual": quality_order,
}

for col, order in ordinal_map.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

print("Ordinal variables classified successfully!")
print(df["ExterQual"].dtype)
df[["ExterQual", "KitchenQual"]].head()

#### Variable summary table

In [ ]:
# OverallQual / OverallCond are stored as numbers (1-10) but are really ordinal.
ordinal_numeric = ["OverallQual", "OverallCond"]
# MSSubClass is a numeric *code* for building type, so it is really nominal.
nominal_numeric = ["MSSubClass"]

def var_type(col):
    if col in ordinal_map or col in ordinal_numeric:
        return "Ordinal"
    if col in nominal_numeric:
        return "Categorical (nominal, stored as number)"
    if col in numeric_cols:
        return "Numerical (discrete)" if df[col].nunique() <= 25 else "Numerical (continuous)"
    return "Categorical (nominal)"

variable_summary = pd.DataFrame({
    "column": df.columns,
    "type": [var_type(c) for c in df.columns],
    "unique_values": [df[c].nunique() for c in df.columns],
    "example_values": [", ".join(map(str, df[c].dropna().unique()[:3])) for c in df.columns],
})

print(variable_summary["type"].value_counts())
pd.set_option("display.max_rows", 100)
display(variable_summary)

### What we learned

- Most columns are either numerical or nominal categorical.
- **`OverallQual` (and `OverallCond`) look numerical but are ordinal**: a quality of 8 is *better* than 4, but not necessarily "twice as good". We keep them as numbers for plots and correlations, but interpret them as ordered grades.
- **`MSSubClass` is a numeric code** for the dwelling type, so averaging it would be meaningless. It is really nominal.
- The numeric "discrete vs continuous" split above uses a simple rule (25 or fewer distinct values = discrete), so it is a good guide, not an exact truth.
- `ExterQual` and `KitchenQual` are now ordered categoricals (Po < Fa < TA < Gd < Ex).

## 3. Descriptive Statistics

We summarise the centre (mean, median, mode) and spread (standard deviation, variance, range, IQR) of the data, plus its shape (skewness, kurtosis).

In [ ]:
df.describe()

#### Statistical calculations

In [ ]:
columns = ["SalePrice", "GrLivArea", "LotArea"]

for col in columns:
    s = df[col]
    print("\nStatistics for:", col)
    print("Mean:", round(s.mean(), 2))
    print("Median:", s.median())
    print("Mode:", s.mode()[0])
    print("Standard Deviation:", round(s.std(), 2))
    print("Variance:", round(s.var(), 2))
    print("Range:", s.max() - s.min())

    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    print("IQR:", Q3 - Q1)
    print("Skewness:", round(s.skew(), 3))
    print("Kurtosis:", round(s.kurt(), 3))

    if s.mean() > s.median():
        print("-> mean > median: RIGHT skewed")
    elif s.mean() < s.median():
        print("-> mean < median: LEFT skewed")
    else:
        print("-> mean = median: symmetric")

#### Categorical columns: counts and mode

In [ ]:
for col in ["Neighborhood", "HouseStyle"]:
    print(f"\n{col}  (mode = {df[col].mode()[0]})")
    print(df[col].value_counts())

In [ ]:
gap = df["SalePrice"].mean() - df["SalePrice"].median()
print(f"SalePrice mean   : {df['SalePrice'].mean():,.0f}")
print(f"SalePrice median : {df['SalePrice'].median():,.0f}")
print(f"Gap (mean - median): {gap:,.0f}  ({gap / df['SalePrice'].median():.1%} of the median)")

### What we learned

- For all three columns, **mean > median** and skewness is positive, so each is **right skewed**: a small number of very expensive / very large houses pull the mean upwards while the median stays near the typical house.
- The gap between mean and median of `SalePrice` (printed above) tells us that **the median is the better "typical price"**, since the mean is dragged up by luxury homes.
- `LotArea` has an extreme kurtosis: a few huge lots make its tails very heavy.
- The most common neighbourhood and house style (the modes) are printed above. Some neighbourhoods have very few houses, which matters later when we compare averages.

## 4. Distribution Analysis

We use histograms with KDE, boxplots, a log transformation, Q-Q plots and countplots to see the shape of `SalePrice` and the key features.

#### SalePrice: histogram + KDE, boxplot and log-transformed histogram

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["SalePrice"], kde=True, color=BASE, ax=axes[0])
axes[0].set_title("Distribution of House Prices")
axes[0].set_xlabel("Sale Price")
axes[0].set_ylabel("Frequency")

sns.boxplot(x=df["SalePrice"], color=BASE, ax=axes[1])
axes[1].set_title("Boxplot of House Prices")
axes[1].set_xlabel("Sale Price")

sns.histplot(np.log1p(df["SalePrice"]), kde=True, color=GREEN, ax=axes[2])
axes[2].set_title("Log Transformed House Prices")
axes[2].set_xlabel("log(1 + Sale Price)")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print("Original Skewness:", round(df["SalePrice"].skew(), 3))
print("Log Skewness     :", round(np.log1p(df["SalePrice"]).skew(), 3))

#### Q-Q plots: how close to a normal distribution?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

stats.probplot(df["SalePrice"], dist="norm", plot=axes[0])
axes[0].set_title("Q-Q plot: SalePrice")

stats.probplot(np.log1p(df["SalePrice"]), dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot: log(SalePrice)")

plt.tight_layout()
plt.show()

#### House quality vs price

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="OverallQual", y="SalePrice", color=BASE)
plt.title("House Quality vs Sale Price")
plt.xlabel("Overall Quality")
plt.ylabel("Sale Price")
plt.gca().yaxis.set_major_formatter(thousands)
plt.show()

q_median = df.groupby("OverallQual")["SalePrice"].median()
q_std    = df.groupby("OverallQual")["SalePrice"].std()
print("Median price rises with every quality step:", q_median.is_monotonic_increasing)
print("\nMedian price by quality:\n", q_median)
print("\nSpread (std) at quality 5 vs quality 10:",
      round(q_std.loc[5]), "vs", round(q_std.loc[10]))

#### Neighbourhood vs price

In [ ]:
order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index

plt.figure(figsize=(16, 6))
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=order, color=BASE)
plt.title("House Prices by Neighborhood (sorted by median price)")
plt.xticks(rotation=90)
plt.gca().yaxis.set_major_formatter(thousands)
plt.show()

nb = df.groupby("Neighborhood")["SalePrice"].agg(["median", "count"]).sort_values("median")
print("Cheapest 3 neighbourhoods (median):\n", nb.head(3))
print("\nMost expensive 3 neighbourhoods (median):\n", nb.tail(3))
print(f"\nMedian price ratio, most expensive / cheapest: {nb['median'].iloc[-1] / nb['median'].iloc[0]:.1f}x")

#### Countplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="OverallQual", color=BASE, ax=axes[0])
axes[0].set_title("House Quality Count")

sns.countplot(data=df, x="HouseStyle", color=BASE, ax=axes[1])
axes[1].set_title("House Style Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### What we learned

1. **`SalePrice` is right skewed** (positive skewness, a long tail of expensive houses, many boxplot outliers on the high side).
2. **A log transform makes it close to normal**: skewness drops from about 1.9 to about 0.1, and the Q-Q plot points follow the straight line much more closely.
3. **Price rises with quality in a clear, ordered way**: the median increases at every step of `OverallQual`, and the spread (std) is much larger at the top grades than at the middle ones.
4. **Neighbourhood matters**: the most expensive neighbourhoods have a median several times higher than the cheapest ones, and their boxes barely overlap with the cheapest group.
5. Most houses are of middling quality (5-7), and one- and two-storey styles dominate the house-style counts, so extreme grades and rare styles rest on few observations.

## 5. Correlation and Covariance

- **Covariance** shows whether two variables move together, but its size depends on their units.
- **Correlation** (Pearson) rescales covariance to the range -1 to +1, so features can be compared fairly.

In this section we compute both and find the top 10 features correlated with `SalePrice`. (`Id` is just a row label, so we drop it.)

In [ ]:
numeric_df = df.select_dtypes(include=np.number).drop(columns="Id")

cov_matrix  = numeric_df.cov()
corr_matrix = numeric_df.corr(method="pearson")

print("Covariance matrix shape :", cov_matrix.shape)
print("Correlation matrix shape:", corr_matrix.shape)

#### Top 10 correlated features

In [ ]:
top_corr = corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)

print("Top 10 Features Correlated with SalePrice (absolute value):")
print(top_corr.head(10).round(3))

print("\nSame features with their sign:")
print(corr_matrix["SalePrice"][top_corr.head(10).index].round(3))

#### Covariance vs correlation for the top 3 features

In [ ]:
top3 = top_corr.head(3).index

for col in top3:
    print("\nFeature:", col)
    print("Covariance with SalePrice :", f"{cov_matrix.loc[col, 'SalePrice']:,.2f}")
    print("Correlation with SalePrice:", round(corr_matrix.loc[col, "SalePrice"], 3))

#### Spearman correlation for ordinal columns

In [ ]:
# Spearman works on ranks, so it suits ordered grades.
# Ordered categoricals are turned into their rank codes (0 = Po ... 4 = Ex).
ordinal_check = pd.DataFrame({
    "OverallQual": df["OverallQual"],
    "ExterQual":   df["ExterQual"].cat.codes,
    "KitchenQual": df["KitchenQual"].cat.codes,
    "SalePrice":   df["SalePrice"],
})

spearman = ordinal_check.corr(method="spearman")["SalePrice"].drop("SalePrice")
pearson_oq = df["OverallQual"].corr(df["SalePrice"])
print("Spearman correlation with SalePrice:\n", spearman.round(3))
print("\nOverallQual: Pearson =", round(pearson_oq, 3), "| Spearman =", round(spearman["OverallQual"], 3))

#### Are the features correlated with each other?

In [ ]:
ga_gc = df["GarageArea"].corr(df["GarageCars"])
print("Correlation between GarageArea and GarageCars:", round(ga_gc, 3))
print("GarageArea with SalePrice:", round(df["GarageArea"].corr(df["SalePrice"]), 3))
print("GarageCars with SalePrice:", round(df["GarageCars"].corr(df["SalePrice"]), 3))

### What we learned

- Covariance values are huge and sit on very different scales, so they cannot be compared across features. Correlation is on a common -1 to +1 scale, which is why it is better for ranking features.
- The top correlates (printed above) are led by **`OverallQual` and `GrLivArea`**. All of the top 10 are positive: bigger, better-built houses sell for more.
- The Spearman correlations for the ordinal grades are also strong, confirming that quality grades relate to price in a consistent *ordered* way, not only linearly.
- **Correlation is not causation.** Classic example: ice-cream sales and drowning both rise in summer, but one does not cause the other. A house with higher quality is often also bigger and in a better area, so we cannot say quality *alone* raises the price.
- **Features also correlate with each other (multicollinearity).** `GarageArea` and `GarageCars` are strongly correlated (printed above) because bigger garages hold more cars, so they carry mostly the *same* information. They should not be counted as two independent reasons for a higher price.

## 6. Scatter Plots and Pair Plots

Scatter plots show the relationship between two numerical variables and make unusual points easy to spot. Pair plots show many pairs at once.

#### Living area vs sale price (unusual houses highlighted)

In [ ]:
unusual = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

plt.figure(figsize=(10, 6))
plt.scatter(df.loc[~unusual, "GrLivArea"], df.loc[~unusual, "SalePrice"],
            color=BASE, alpha=0.5, label="Other houses")
plt.scatter(df.loc[unusual, "GrLivArea"], df.loc[unusual, "SalePrice"],
            color=ACCENT, s=90, edgecolor="black", label="GrLivArea > 4000 and price < 300k")

for _, row in df[unusual].iterrows():
    plt.annotate(f"Id {int(row['Id'])}", (row["GrLivArea"], row["SalePrice"]),
                 xytext=(-45, 12), textcoords="offset points")

plt.title("Living Area vs Sale Price")
plt.xlabel("Ground Living Area (sq ft)")
plt.ylabel("Sale Price")
plt.gca().yaxis.set_major_formatter(thousands)
plt.legend()
plt.show()

#### Basement area vs sale price

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="TotalBsmtSF", y="SalePrice", color=BASE, alpha=0.6)
plt.title("Basement Area vs Sale Price")
plt.xlabel("Total Basement Area (sq ft)")
plt.ylabel("Sale Price")
plt.gca().yaxis.set_major_formatter(thousands)
plt.show()

#### Year built vs sale price

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="YearBuilt", y="SalePrice", color=BASE, alpha=0.6)
plt.title("Year Built vs Sale Price")
plt.xlabel("Year Built")
plt.ylabel("Sale Price")
plt.gca().yaxis.set_major_formatter(thousands)
plt.show()

#### Pair plot (coloured by central air conditioning)

In [ ]:
selected = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]

sns.pairplot(
    df[selected + ["CentralAir"]],
    hue="CentralAir",
    diag_kind="kde",
    palette={"Y": BASE, "N": ACCENT},
    plot_kws={"alpha": 0.5},
    corner=True,
)
plt.suptitle("Pair plot of selected features (by CentralAir)", y=1.02)
plt.show()

### What we learned

1. **Larger living areas generally mean higher prices**: the cloud rises from left to right.
2. **Basement area shows a similar rising relationship**, with a visible cluster at zero (houses with no basement).
3. **Newer houses tend to sell for higher prices**, but the relationship is looser than for area, so build year is a weaker predictor.
4. In the living-area plot **two houses stand out**: very large (over 4000 sq ft) but sold for a relatively low price. They sit far below the trend and will pull a fitted trend line down. We investigate them in Step 8.
5. In the pair plot, houses **without central air (N)** are concentrated among the older, cheaper homes, so central air partly acts as a marker for age and price level.

## 7. Correlation Heatmap

A heatmap puts many correlations in one picture. We use the top 10 features correlated with `SalePrice` and hide the duplicate upper half.

In [ ]:
top10_features = ["SalePrice"] + top_corr.head(10).index.tolist()    # SalePrice first, then top 10 features
corr_top = df[top10_features].corr()

mask = np.triu(np.ones_like(corr_top, dtype=bool))

plt.figure(figsize=(12, 8))
sns.heatmap(corr_top, annot=True, fmt=".2f", cmap="coolwarm", center=0, mask=mask)
plt.title("Top 10 Features Correlated with SalePrice")
plt.show()

### What we learned

- The bottom row/column for `SalePrice` shows that the strongest relationships with price are with quality and size features.
- Several features are **strongly correlated with each other**, for example `GarageCars` with `GarageArea`, and `TotalBsmtSF` with `1stFlrSF`. They overlap in the information they give about a house.
- A heatmap only shows *linear* association. It does not prove cause and effect.

## 8. Outlier Detection

An outlier is a value very different from most others. We use two methods and compare them:

1. **IQR method**: flags values beyond 1.5 x IQR from the quartiles.
2. **Z-score method**: flags values more than 3 standard deviations from the mean.

In [ ]:
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)].index


def zscore_outliers(series, threshold=3):
    clean = series.dropna()
    z = pd.Series(stats.zscore(clean), index=clean.index)
    return z[abs(z) > threshold].index

In [ ]:
for column in ["GrLivArea", "SalePrice"]:
    iqr_result = iqr_outliers(df[column])
    z_result = zscore_outliers(df[column])

    print("\nColumn:", column)
    print("IQR Outliers     :", len(iqr_result))
    print("Z-score Outliers :", len(z_result))
    print("Common (overlap) :", len(set(iqr_result) & set(z_result)))
    print(f"Skewness: {df[column].skew():.2f}")

#### Boxplots with outliers marked

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=df["GrLivArea"], color=BASE, ax=axes[0])
axes[0].set_title("Outliers in Ground Living Area")

sns.boxplot(y=df["SalePrice"], color=BASE, ax=axes[1])
axes[1].set_title("Outliers in Sale Price")
axes[1].yaxis.set_major_formatter(thousands)

plt.tight_layout()
plt.show()

#### Scatter plot: which houses do the two methods flag?

In [ ]:
flagged = set(iqr_outliers(df["GrLivArea"])) | set(iqr_outliers(df["SalePrice"])) \
        | set(zscore_outliers(df["GrLivArea"])) | set(zscore_outliers(df["SalePrice"]))
is_flagged = df.index.isin(flagged)
extreme = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

plt.figure(figsize=(10, 6))
plt.scatter(df.loc[~is_flagged, "GrLivArea"], df.loc[~is_flagged, "SalePrice"],
            color=BASE, alpha=0.4, label="Not flagged")
plt.scatter(df.loc[is_flagged & ~extreme, "GrLivArea"], df.loc[is_flagged & ~extreme, "SalePrice"],
            color="#f2cc8f", edgecolor="black", label="Flagged by IQR or z-score")
plt.scatter(df.loc[extreme, "GrLivArea"], df.loc[extreme, "SalePrice"],
            color=ACCENT, s=100, edgecolor="black", label="Very large, low price")

plt.xlabel("Ground Living Area")
plt.ylabel("Sale Price")
plt.title("Outlier Detection in House Prices")
plt.gca().yaxis.set_major_formatter(thousands)
plt.legend()
plt.show()

print("Total flagged houses:", len(flagged))

#### Decision: error, unusual sale, or genuine sale?

In [ ]:
extreme_idx = df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)].index

print("Very large houses with a low price:")
display(df.loc[extreme_idx, ["Id", "GrLivArea", "SalePrice", "OverallQual", "Neighborhood", "SaleCondition"]])

print("Other very large houses (over 4000 sq ft) for comparison:")
display(df[(df["GrLivArea"] > 4000) & (df["SalePrice"] >= 300000)]
        [["Id", "GrLivArea", "SalePrice", "OverallQual", "SaleCondition"]])

### Decision and interpretation

- **Remove:** the very large, low-priced houses above. Their `SaleCondition` is **Partial** (the house was sold before it was completed), and a price far below what their size and quality would suggest is not a normal market sale. They are *unusual sales*, not typical houses, and they distort the size-price trend. They are removed in Step 9 and the decision is recorded there.
- **Keep:** all the other flagged points. The expensive houses and the large expensive houses are real sales at the top of the market. Removing them would hide real information, so we keep them and just remember that they stretch the right tail.
- **IQR vs z-score:** the IQR method flags more points than the z-score method on skewed variables. The reason is that the extreme values themselves **inflate the mean and standard deviation**, which pushes the z-score cutoff further out (the outliers "hide" each other). The quartiles used by IQR are not pulled by the extremes.
- Outliers should never be removed automatically: first ask whether each one is an error or a real, if unusual, sale.

## 9. Data Quality Checks

We build a separate cleaned copy, `df_clean`, and leave the raw `df` untouched. Checks:

1. Remove the unusual sales identified in Step 8.
2. Find duplicate rows.
3. Check categorical values for inconsistent labels.
4. Handle missing values, **documenting every decision**.
5. Check for impossible values.

In [ ]:
# Step 8 decision: remove the very large, low-priced (Partial) sales
df_clean = df.drop(index=extreme_idx).copy()
print("Shape after removing unusual sales:", df_clean.shape)

# Duplicates: 'Id' is unique for every row, so compare all the OTHER columns.
dup_mask = df_clean.drop(columns="Id").duplicated()
print("Duplicate rows (ignoring Id):", dup_mask.sum())

if dup_mask.sum() > 0:
    df_clean = df_clean.loc[~dup_mask]
    print("Shape after removing duplicates:", df_clean.shape)

#### Checking categorical columns for inconsistent labels

In [ ]:
# Remove extra spaces (ordered categoricals are skipped so the ordering is kept)
text_cols = [c for c in df_clean.select_dtypes(exclude=np.number).columns
             if not isinstance(df_clean[c].dtype, pd.CategoricalDtype)]

for c in text_cols:
    df_clean[c] = df_clean[c].str.strip()

# Casing / spacing variants of the same label inside one column
found = False
for c in text_cols:
    vals = df_clean[c].dropna().unique()
    if len({str(v).lower().replace(" ", "") for v in vals}) != len(vals):
        print("Case/space variants in", c, sorted(vals))
        found = True
if not found:
    print("No casing or spacing variants found inside any single column.")

# Same material spelled differently across two related columns?
e1 = set(df_clean["Exterior1st"].dropna())
e2 = set(df_clean["Exterior2nd"].dropna())
print("\nOnly in Exterior1st:", sorted(e1 - e2))
print("Only in Exterior2nd:", sorted(e2 - e1))

In [ ]:
# The same materials are spelled differently in Exterior2nd. Standardise to Exterior1st's spelling.
exterior_fix = {"CmentBd": "CemntBd", "Wd Shng": "WdShing", "Brk Cmn": "BrkComm"}
df_clean["Exterior2nd"] = df_clean["Exterior2nd"].replace(exterior_fix)

e2 = set(df_clean["Exterior2nd"].dropna())
print("Only in Exterior1st:", sorted(e1 - e2))
print("Only in Exterior2nd:", sorted(e2 - e1))

#### Missing values: what does a missing value *mean*?

In this dataset the data description says that for many columns **NA means "the feature does not exist"** (no pool, no garage, no basement...). Those are not missing data, so we fill them with the label `"None"` instead of dropping them.

Also note: pandas treats the text `"None"` as a missing value when reading a CSV. `MasVnrType` really uses `"None"` as a category (no masonry veneer), so most of its "missing" values are genuine "None" houses.

In [ ]:
decisions = []   # (column, number missing, what it means, action)

# 1) NA means the feature is absent -> "None"
absent = {
    "PoolQC": "no pool",          "MiscFeature": "no misc feature",
    "Alley": "no alley access",   "Fence": "no fence",
    "FireplaceQu": "no fireplace",
    "GarageType": "no garage",    "GarageFinish": "no garage",
    "GarageQual": "no garage",    "GarageCond": "no garage",
    "BsmtQual": "no basement",    "BsmtCond": "no basement",
    "BsmtExposure": "no basement", "BsmtFinType1": "no basement",
    "BsmtFinType2": "no basement",
}
for col, meaning in absent.items():
    n = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna("None")
    decisions.append((col, n, meaning, 'fill with "None"'))

# 2) Masonry veneer: "None" was read as NaN by pandas
n = df_clean["MasVnrType"].isnull().sum()
df_clean["MasVnrType"] = df_clean["MasVnrType"].fillna("None")
decisions.append(("MasVnrType", n, 'text "None" read as NaN (no veneer)', 'fill with "None"'))

n = df_clean["MasVnrArea"].isnull().sum()
df_clean["MasVnrArea"] = df_clean["MasVnrArea"].fillna(0)
decisions.append(("MasVnrArea", n, "no veneer -> area 0", "fill with 0"))

# 3) Electrical: a single genuinely missing value -> most common type
n = df_clean["Electrical"].isnull().sum()
df_clean["Electrical"] = df_clean["Electrical"].fillna(df_clean["Electrical"].mode()[0])
decisions.append(("Electrical", n, "truly missing", "fill with the mode"))

# 4) GarageYrBlt: houses without a garage have no garage year
bad = df_clean["GarageYrBlt"] > df_clean["YrSold"]            # impossible values (if any)
df_clean.loc[bad, "GarageYrBlt"] = np.nan
n = df_clean["GarageYrBlt"].isnull().sum()
df_clean["GarageYrBlt"] = df_clean["GarageYrBlt"].fillna(df_clean["YearBuilt"])
decisions.append(("GarageYrBlt", n, "no garage", "fill with YearBuilt (assumption; use with care)"))

# 5) LotFrontage: truly missing -> median of the same neighbourhood
n = df_clean["LotFrontage"].isnull().sum()
df_clean["LotFrontage"] = df_clean.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
df_clean["LotFrontage"] = df_clean["LotFrontage"].fillna(df_clean["LotFrontage"].median())
decisions.append(("LotFrontage", n, "truly missing", "median by Neighborhood (then overall median)"))

display(pd.DataFrame(decisions, columns=["column", "n_missing_filled", "meaning", "action"]))

In [ ]:
# Safety net: anything still missing (there should be nothing)
left = df_clean.isnull().sum()
left = left[left > 0]
print("Still missing after documented decisions:", left.to_dict())

for c in left.index:
    if pd.api.types.is_numeric_dtype(df_clean[c]):
        df_clean[c] = df_clean[c].fillna(df_clean[c].median())
    else:
        df_clean[c] = df_clean[c].fillna("Unknown")

print("\nRemaining missing values:", df_clean.isnull().sum().sum())
print("Dataset Shape:", df_clean.shape)

#### Ordinal columns in the cleaned data

In [ ]:
# Now that 'absent' is a real label, build the ordered categories (None < Po < Fa < TA < Gd < Ex)
with_none = ["None"] + quality_order
ordinal_clean = {
    "ExterQual": quality_order, "ExterCond": quality_order,
    "HeatingQC": quality_order, "KitchenQual": quality_order,
    "BsmtQual": with_none, "BsmtCond": with_none,
    "FireplaceQu": with_none, "GarageQual": with_none,
    "GarageCond": with_none, "PoolQC": with_none,
}
for col, order in ordinal_clean.items():
    df_clean[col] = pd.Categorical(df_clean[col].astype(str), categories=order, ordered=True)

print("Ordinal columns with unrecognised labels:",
      {c: int(df_clean[c].isnull().sum()) for c in ordinal_clean if df_clean[c].isnull().sum() > 0})
print("Remaining missing values:", df_clean.isnull().sum().sum())

#### Check impossible values

In [ ]:
print("YearBuilt greater than YrSold   :", (df_clean["YearBuilt"] > df_clean["YrSold"]).sum())
print("YearRemodAdd before YearBuilt   :", (df_clean["YearRemodAdd"] < df_clean["YearBuilt"]).sum())
print("GarageYrBlt greater than YrSold :", (df_clean["GarageYrBlt"] > df_clean["YrSold"]).sum())

for column in ["GrLivArea", "TotalBsmtSF", "LotArea", "GarageArea"]:
    print(column, "- negative values:", (df_clean[column] < 0).sum())

### What we learned

- **No duplicate rows and no impossible values** were found (checks are kept so the notebook works on any version of the data).
- **Inconsistent labels were found**: the same exterior materials are spelled differently in `Exterior1st` and `Exterior2nd` (`CemntBd` vs `CmentBd`, `WdShing` vs `Wd Shng`, `BrkComm` vs `Brk Cmn`). Stripping spaces would not catch this, so a mapping dictionary was used.
- **Most "missing" values are not missing.** Pool, alley, fence, misc feature, fireplace, garage and basement columns are blank because the feature does not exist, so they were labelled `"None"` and kept. Only `LotFrontage` and `Electrical` are genuinely missing, and each has a documented fill rule.
- `MasVnrType` was another trap: pandas converted the real label `"None"` into NaN.
- Every decision is listed in the decision table above, and the raw `df` was never changed.

## 10. Final EDA Dashboard

The dashboard combines four visualisations from the cleaned data:

1. Average sale price by year built.
2. Top 10 neighbourhoods by average sale price.
3. Sale price distribution with mean and median.
4. Ground living area vs sale price, coloured by overall quality.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Average price by year built (Matplotlib line plot)
year_price = df_clean.groupby("YearBuilt")["SalePrice"].mean()
axes[0, 0].plot(year_price.index, year_price.values, color=BASE, label="Average price")
axes[0, 0].set_title("Average Sale Price by Year Built")
axes[0, 0].set_xlabel("Year Built")
axes[0, 0].set_ylabel("Average Sale Price")
axes[0, 0].yaxis.set_major_formatter(thousands)
axes[0, 0].legend()

# Panel 2: Top 10 neighbourhoods (Matplotlib horizontal bars)
nb = df_clean.groupby("Neighborhood")["SalePrice"].agg(["mean", "count"])
top10_nb = nb.nlargest(10, "mean").sort_values("mean")
labels = [f"{name} (n={int(row['count'])})" for name, row in top10_nb.iterrows()]
axes[0, 1].barh(labels, top10_nb["mean"].values, color=BASE)
axes[0, 1].set_title("Top 10 Neighborhoods by Average Price")
axes[0, 1].set_xlabel("Average Sale Price")
axes[0, 1].set_ylabel("Neighborhood (n = number of houses)")
axes[0, 1].xaxis.set_major_formatter(thousands)

# Panel 3: Distribution with mean and median (Seaborn)
sns.histplot(df_clean["SalePrice"], kde=True, color=BASE, ax=axes[1, 0])
mean_price = df_clean["SalePrice"].mean()
median_price = df_clean["SalePrice"].median()
axes[1, 0].axvline(mean_price, color=ACCENT, linestyle="--", linewidth=2, label=f"Mean ({mean_price/1000:,.0f}k)")
axes[1, 0].axvline(median_price, color=GREEN, linestyle=":", linewidth=2, label=f"Median ({median_price/1000:,.0f}k)")
axes[1, 0].set_title("Sale Price Distribution")
axes[1, 0].set_xlabel("Sale Price")
axes[1, 0].xaxis.set_major_formatter(thousands)
axes[1, 0].legend()

# Panel 4: Living area vs price, coloured by quality
scatter = axes[1, 1].scatter(df_clean["GrLivArea"], df_clean["SalePrice"],
                             c=df_clean["OverallQual"], cmap="viridis", alpha=0.7)
axes[1, 1].set_title("Living Area vs Sale Price")
axes[1, 1].set_xlabel("Ground Living Area (sq ft)")
axes[1, 1].set_ylabel("Sale Price")
axes[1, 1].yaxis.set_major_formatter(thousands)
fig.colorbar(scatter, ax=axes[1, 1], label="Overall Quality")

fig.suptitle("House Price EDA Summary", fontsize=16)
plt.tight_layout()
plt.savefig("eda_dashboard.png", dpi=300, bbox_inches="tight")
plt.show()

print("Dashboard saved successfully as eda_dashboard.png")

### What we learned

- Newer houses tend to sell for more, with a lot of year-to-year noise in the early decades where there are few houses per year.
- A handful of neighbourhoods clearly lead on average price. The `n =` in the labels warns us that averages based on few houses are less reliable.
- The mean sits to the right of the median, which is the right-skew from Step 3.
- In the scatter plot, colour moves from dark (low quality) to bright (high quality) as price rises, so **both size and quality push price up**.

## 11. Verifying the Expected Insights

The project asked us to **verify** six expected insights rather than assume them. The cell below recomputes each one from the data.

In [ ]:
print("1. SalePrice is right skewed; log makes it near normal")
print(f"   skewness raw = {df['SalePrice'].skew():.2f} | log = {np.log1p(df['SalePrice']).skew():.2f}")

print("\n2. OverallQual and GrLivArea are among the strongest correlates")
print("   top 5:", top_corr.head(5).round(2).to_dict())

print("\n3. Price rises steadily with quality, spread is larger at the top")
print("   median rises at every grade:", df.groupby('OverallQual')['SalePrice'].median().is_monotonic_increasing)
print("   std at grade 5 vs 10:", round(q_std.loc[5]), "vs", round(q_std.loc[10]))

nb_med = df.groupby("Neighborhood")["SalePrice"].median().sort_values()
print("\n4. A few neighbourhoods have clearly higher medians")
print(f"   cheapest median = {nb_med.iloc[0]:,.0f} | most expensive median = {nb_med.iloc[-1]:,.0f}")
print("   top 3:", list(nb_med.index[-3:]))

print("\n5. A handful of very large, low-price houses distort trend lines")
no_out = df.drop(index=extreme_idx)
print(f"   houses found: {len(extreme_idx)} | corr(GrLivArea, SalePrice) with them: "
      f"{df['GrLivArea'].corr(df['SalePrice']):.3f} | without them: {no_out['GrLivArea'].corr(no_out['SalePrice']):.3f}")

print("\n6. Many apparent missing values are really 'feature absent'")
print(f"   columns with NA in raw data: {len(missing_table)} | of which 'feature absent': {len(absent) + 1}")

## 12. Conclusion

In this project we performed Exploratory Data Analysis on the Ames Housing dataset (1460 houses, 81 columns) and answered the six guiding questions:

1. **Distribution of price:** `SalePrice` is right skewed (mean above median). A log transform brings it close to a normal shape.
2. **Strongest correlates:** overall quality (`OverallQual`) and living area (`GrLivArea`) lead, followed by garage, basement and bathroom size features.
3. **Quality and price:** price rises steadily with quality grade, and the spread of prices widens at the top grades.
4. **Neighbourhood:** a few neighbourhoods have clearly higher median prices than the rest.
5. **Outliers:** two very large, low-priced houses are *Partial* sales, so we treated them as unusual sales and removed them. Other high-price outliers are genuine and were kept.
6. **Missing values:** most missing values mean "feature absent" (pool, alley, fence, garage, basement...), so they were kept as `"None"`. Only a couple of columns were genuinely missing and were filled with documented rules.

We also found inconsistent spellings in the exterior-material columns and avoided a hidden trap where pandas reads the label `"None"` as a missing value.

### Limitations

- The data covers **one city (Ames, Iowa) in a short time window**, so conclusions may not apply to other markets.
- **Correlation is not causation**: size, quality and neighbourhood are linked to each other, so we cannot say how much any single factor adds to price.
- Pearson correlation only captures **linear** relationships.
- **Cleaning choices involve assumptions** (for example, filling `GarageYrBlt` with `YearBuilt`, and removing the two Partial sales). Different choices could change the numbers.
- Averages for **neighbourhoods with few houses** are less reliable.
- This is exploratory work only: no model was built, so we make **no predictions**.